# **Reddit Data Collection**

This notebook collects GameStop-related Reddit posts from r/wallstreetbets using the Arctic Shift API. The collected post-level data provide the Reddit information source used in subsequent sentiment analysis, topic modeling, and integration with GME market data.

The collection period spans December 1, 2020 through March 31, 2021. The collection process uses batch retrieval, retry logic, and checkpointing to support reliable data acquisition and recovery from temporary API failures.

## **1. Setup**

Required Python libraries are imported for API requests, tabular data processing, request timing, and file management.

In [ ]:
# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------

# Standard library
import time
from pathlib import Path

# Data manipulation
import pandas as pd

# Web requests
import requests


## **2. Data Collection Configuration**

The API request is configured to retrieve GameStop-related posts from r/wallstreetbets using the search terms `GME GameStop`. The requested collection period extends from December 1, 2020 through March 31, 2021.

Posts are retrieved in batches of up to 50 records. Retry parameters are also defined to handle temporary API failures during the collection process.

In [ ]:
# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

URL = "https://arctic-shift.photon-reddit.com/api/posts/search"

SUBREDDIT = "wallstreetbets"
QUERY = "GME GameStop"

START_DATE = "2020-12-01"
END_DATE = "2021-03-31"

BATCH_SIZE = 50
MAX_RETRIES = 5
RETRY_DELAY = 20

SORT = 'asc'

## **3. API Request Setup**

The collection state and requested Reddit fields are initialized before querying the API. Each retrieved post includes its identifier, creation timestamp, title, self-text, score, number of comments, and URL.

In [29]:
all_posts = []

before = END_DATE
batch_number = 1

## **4. Batch Retrieval and API Validation**

A reusable batch-retrieval function is defined to request Reddit posts using the configured search criteria. The function includes retry logic for temporary connection or API errors and pauses between failed attempts before terminating after the maximum number of retries.

In [30]:
FIELDS = ",".join([
    "id",
    "created_utc",
    "title",
    "selftext",
    "score",
    "num_comments",
    "url"
])

In [38]:
def fetch_batch(after):

    params = {
        "subreddit": SUBREDDIT,
        "query": QUERY,
        "after": after,
        "before": END_DATE,
        "limit": BATCH_SIZE,
        "fields": FIELDS,
        "sort": SORT
    }

    for attempt in range(MAX_RETRIES):

        try:
            response = requests.get(
                URL,
                params=params,
                timeout=30
            )

            if response.status_code == 200:
                return response.json()["data"]

            print(
                f"Attempt {attempt + 1} failed:",
                response.status_code,
                response.json()
            )

        except requests.exceptions.RequestException as e:
            print(
                f"Attempt {attempt + 1} connection error:",
                type(e).__name__,
                str(e)
            )

        if attempt < MAX_RETRIES - 1:
            time.sleep(RETRY_DELAY)

    return None

### **4.1 Test API Request**

A single batch is retrieved before the full collection process to verify that the API request is successful and that the returned records contain the expected fields.

In [39]:
test_batch = fetch_batch(START_DATE)

print(type(test_batch))
print(len(test_batch))

<class 'list'>
50


In [40]:
pd.DataFrame(test_batch).head()

,created_utc,id,num_comments,score,selftext,title,url
0,1606793922,k4csaa,936,2543,>[Oh and uh short burn of the century comin so...,The REAL Greatest Short Burn of the Century Pa...,https://www.reddit.com/r/wallstreetbets/commen...
1,1607010382,k5zmcd,303,2251,"You've been holding GME for weeks, having dump...",Exactly how the GME squeeze will go for you.,https://www.reddit.com/r/wallstreetbets/commen...
2,1607031631,k66w5h,327,785,"GME has a new logo, graphics and slogan on all...",GME rebranding in progress - they've just chan...,https://www.reddit.com/r/wallstreetbets/commen...
3,1607042510,k6aa8p,0,1,[deleted],(GME) Gamestop's new branding looks strangely ...,https://i.redd.it/wkmgr34vg2361.jpg
4,1607043622,k6akre,0,1,[deleted],GME - GameStop's new branding looks strangely ...,https://i.redd.it/f90srt4lj2361.jpg


## **5. Full Reddit Data Collection**

The full collection process retrieves matching posts sequentially across the study period. After each successful batch, collected records are deduplicated by post ID, sorted chronologically, and saved to a checkpoint file.

If an existing checkpoint is detected, collection resumes from the most recent successfully saved post rather than restarting from the beginning. This design reduces data loss and unnecessary API requests when temporary failures interrupt the collection process.

In [43]:
all_posts = []
after = START_DATE
batch_number = 1

# Checkpoint file for resuming data collection
CHECKPOINT_PATH = Path("../data/raw/reddit_checkpoint.csv")

# Resume from checkpoint if it already exists
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)

    all_posts = checkpoint_df.to_dict("records")
    after = int(checkpoint_df["created_utc"].max())

    print(
        f"Resuming from checkpoint: "
        f"{len(all_posts)} posts already collected."
    )

else:
    all_posts = []
    after = START_DATE

    print("No checkpoint found. Starting new collection.")

batch_number = 1


while True:
    print(f"\nBatch {batch_number} | after = {after}")

    batch = fetch_batch(after)

    # Stop if API fails after all retries
    if batch is None:
        print(
            "Collection paused: API failed after maximum retries. "
            "Progress has been saved."
        )
        break

    # Stop if there are no more matching posts
    if len(batch) == 0:
        print("No more posts found.")
        break

    all_posts.extend(batch)

    batch_df = pd.DataFrame(batch)

    oldest_timestamp = int(batch_df["created_utc"].min())
    newest_timestamp = int(batch_df["created_utc"].max())

    oldest_date = pd.to_datetime(
        oldest_timestamp,
        unit="s",
        utc=True
    )

    newest_date = pd.to_datetime(
        newest_timestamp,
        unit="s",
        utc=True
    )

    # Save checkpoint after every successful batch
    checkpoint_df = pd.DataFrame(all_posts)

    checkpoint_df = (
        checkpoint_df
        .drop_duplicates(subset="id")
        .sort_values("created_utc")
        .reset_index(drop=True)
    )

    checkpoint_df.to_csv(
        CHECKPOINT_PATH,
        index=False
    )

    # Keep in-memory data consistent with saved checkpoint
    all_posts = checkpoint_df.to_dict("records")

    print(
        f"Collected {len(batch)} posts | "
        f"{oldest_date} → {newest_date} | "
        f"Total saved: {len(checkpoint_df)}"
    )

    # Move forward from the latest successfully saved post
    after = int(checkpoint_df["created_utc"].max())

    if len(batch) < BATCH_SIZE:
        print("Reached final batch.")
        break

    batch_number += 1

    # Pause between requests
    time.sleep(10)

Resuming from checkpoint: 300 posts already collected.

Batch 1 | after = 1611415125
Attempt 1 failed: 422 {'data': None, 'error': 'Timeout. Maybe slow down a bit'}
Attempt 2 failed: 422 {'data': None, 'error': 'Timeout. Maybe slow down a bit'}
Attempt 3 failed: 422 {'data': None, 'error': 'Timeout. Maybe slow down a bit'}
Collected 50 posts | 2021-01-23 15:39:02+00:00 → 2021-01-26 00:09:34+00:00 | Total saved: 350

Batch 2 | after = 1611619774
Attempt 1 failed: 422 {'data': None, 'error': 'Timeout. Maybe slow down a bit'}
Attempt 2 failed: 422 {'data': None, 'error': 'Timeout. Maybe slow down a bit'}
Collected 50 posts | 2021-01-26 00:22:43+00:00 → 2021-01-26 22:29:53+00:00 | Total saved: 400

Batch 3 | after = 1611700193
Collected 50 posts | 2021-01-26 22:37:09+00:00 → 2021-01-27 11:55:57+00:00 | Total saved: 450

Batch 4 | after = 1611748557
Attempt 1 failed: 422 {'data': None, 'error': 'Timeout. Maybe slow down a bit'}
Attempt 2 failed: 422 {'data': None, 'error': 'Timeout. Maybe s

## **6. Data Validation and Coverage Assessment**

The completed checkpoint dataset is reloaded and validated before export. Validation checks examine dataset dimensions, temporal coverage, duplicate post IDs, missing values, and the distribution of collected posts across the study period.

In [47]:
reddit_raw = pd.read_csv(CHECKPOINT_PATH)

print("Shape:", reddit_raw.shape)

reddit_raw["created_utc_dt"] = pd.to_datetime(
    reddit_raw["created_utc"],
    unit="s",
    utc=True
)

print("Date range:")
print(reddit_raw["created_utc_dt"].min())
print(reddit_raw["created_utc_dt"].max())

print("Duplicate IDs:", reddit_raw["id"].duplicated().sum())

print("\nMissing values:")
print(
    reddit_raw.isna().sum()
)

Shape: (2439, 7)
Date range:
2020-12-01 03:38:42+00:00
2021-03-30 13:04:04+00:00
Duplicate IDs: 0

Missing values:
created_utc         0
id                  0
num_comments        0
score               0
selftext          510
title               0
url                 0
created_utc_dt      0
dtype: int64


### **6.1 Temporal Coverage**

In [45]:
reddit_raw["created_utc_dt"].dt.date.value_counts().sort_index()

created_utc_dt
2020-12-01    1
2020-12-03    2
2020-12-04    4
2020-12-05    2
2020-12-06    3
             ..
2021-03-26    4
2021-03-27    4
2021-03-28    6
2021-03-29    4
2021-03-30    3
Name: count, Length: 115, dtype: int64

In [46]:
all_dates = pd.date_range(
    start=reddit_raw["created_utc_dt"].min().normalize(),
    end=reddit_raw["created_utc_dt"].max().normalize(),
    freq="D",
    tz="UTC"
)

dates_with_posts = pd.DatetimeIndex(
    reddit_raw["created_utc_dt"].dt.normalize().unique()
)

missing_dates = all_dates.difference(dates_with_posts)

print("Number of calendar days:", len(all_dates))
print("Days with matching posts:", len(dates_with_posts))
print("Days without matching posts:", len(missing_dates))
print("\nDates without posts:")
print(missing_dates)

Number of calendar days: 120
Days with matching posts: 115
Days without matching posts: 5

Dates without posts:
DatetimeIndex(['2020-12-02 00:00:00+00:00', '2020-12-17 00:00:00+00:00',
               '2020-12-18 00:00:00+00:00', '2020-12-19 00:00:00+00:00',
               '2021-01-07 00:00:00+00:00'],
              dtype='datetime64[s, UTC]', freq=None)


### **Coverage Summary**

The collected dataset contains Reddit posts on **115 of the 120 calendar days** between December 1, 2020 and March 30, 2021. Five dates contain no posts matching the specified search criteria.

The absence of matching posts on these dates is retained as part of the observed Reddit activity pattern rather than treated as missing records.

## **7. Export Raw Reddit Data**

The validated Reddit dataset is exported as a raw CSV file for subsequent text cleaning, deduplication, and feature preparation in `04_data_preprocessing.ipynb`.

In [48]:
RAW_PATH = "../data/raw/reddit/wsb_gme_archive.csv"

reddit_raw.drop(columns=["created_utc_dt"]).to_csv(
    RAW_PATH,
    index=False
)

print(f"Saved {len(reddit_raw)} raw Reddit posts to {RAW_PATH}")

Saved 2439 raw Reddit posts to ../data/raw/reddit/wsb_gme_archive.csv


### **Collection Summary**

The final raw Reddit dataset contains **2,439 GameStop-related posts** collected from r/wallstreetbets between **December 1, 2020 and March 30, 2021**. No duplicate post IDs were identified, while missing values were limited to the `selftext` field for posts without available body text.

The raw dataset is subsequently cleaned and transformed in `04_data_preprocessing.ipynb`, where post titles and available self-text are prepared for sentiment and topic analyses and exact same-day duplicate text is removed.